# Pyr Root-Level Nuclei Curation Exploration

This notebook records my exploratory/manual curation workflow for building a root-level nuclei table from the local `c3_nuclei_v1` Parquet file and screening obvious vascular-like roots from a broader working population. It works at the `pt_root_id` level, not at the individual nucleus-row level.

Root-level features such as nucleus count, volume summaries, and spatial extent are used to generate vascular candidates for visual review. I then used manual inspection in the Pyr.ai volume to decide which candidate roots were clear vascular segmentations and should be removed.

This is not an automated or validated vascular classifier. The notebook produces a screened nonvascular working dataset for downstream analysis, and when local output artifacts already exist it may preserve them rather than overwrite them.

## Inputs

This notebook starts from local artifacts: the nuclei Parquet table generated in the previous notebook and a CA3 annotation CSV used as reference metadata.

In [1]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root(start=Path.cwd().resolve()):
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "helpers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError(
        "Could not find the Pyr project root. Expected a parent directory containing "
        "notebooks/helpers and data."
    )


project_root = find_project_root()
helpers_dir = project_root / "notebooks" / "helpers"
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from path_behavior import format_path, print_path

# Set to True to show full absolute paths in user-facing notebook output.
show_full_path = False
overwrite_existing = False

nuclei_parquet_path = project_root / "data" / "nuclei" / "c3_nuclei_v1_mat195.parquet"
annotation_csv_path = project_root / "data" / "mouse_hippocampus_ca3_cell_annotations_export.csv"

print_path("nuclei Parquet", nuclei_parquet_path, project_root, show_full_path)
print_path("annotation CSV", annotation_csv_path, project_root, show_full_path)


nuclei Parquet: data\nuclei\c3_nuclei_v1_mat195.parquet
annotation CSV: data\mouse_hippocampus_ca3_cell_annotations_export.csv


## Load Local Tables

Load the local nuclei table and annotation CSV. This notebook does not query CAVE directly.

In [2]:
nuclei_df = pd.read_parquet(nuclei_parquet_path)
annotation_df = pd.read_csv(annotation_csv_path)

print(f"nuclei_df shape: {nuclei_df.shape}")
print(f"annotation_df shape: {annotation_df.shape}")
print("nuclei columns:")
print(nuclei_df.columns.tolist())
print("annotation columns:")
print(annotation_df.columns.tolist())


nuclei_df shape: (35499, 10)
annotation_df shape: (2764, 10)
nuclei columns:
['id', 'created', 'superceded_id', 'valid', 'volume', 'pt_supervoxel_id', 'pt_root_id', 'pt_position', 'bb_start_position', 'bb_end_position']
annotation columns:
['Cell ID', 'SV ID', 'x', 'y', 'z', 'type', 'subtypes', 'outputs', 'inputs', 'Links']


## Exclude Root ID 0

`pt_root_id == 0` rows are excluded before root-level curation because they are not assigned to a usable segmentation root.

In [3]:
# Exclude nuclei rows without an assigned segmentation root before root-level curation.
root0_row_count = int((nuclei_df['pt_root_id'] == 0).sum())
nuclei_nonzero = nuclei_df[nuclei_df['pt_root_id'] != 0].copy()

print(f"rows excluded with pt_root_id == 0: {root0_row_count}")
print(f"nonzero nuclei rows retained for root-level aggregation: {len(nuclei_nonzero)}")
print(f"unique nonzero pt_root_id count: {nuclei_nonzero['pt_root_id'].nunique(dropna=True)}")


rows excluded with pt_root_id == 0: 11245
nonzero nuclei rows retained for root-level aggregation: 24254
unique nonzero pt_root_id count: 13873


## Temporary Position Expansion

Expand `pt_position` into temporary `x`, `y`, and `z` columns so spatial ranges can be calculated for each root.

In [4]:
def position_to_xyz(value):
    try:
        values = list(value)
    except TypeError:
        return pd.Series({'x': pd.NA, 'y': pd.NA, 'z': pd.NA})
    if len(values) != 3:
        return pd.Series({'x': pd.NA, 'y': pd.NA, 'z': pd.NA})
    return pd.Series({'x': values[0], 'y': values[1], 'z': values[2]})

position_xyz = nuclei_nonzero['pt_position'].apply(position_to_xyz)
for axis in ['x', 'y', 'z']:
    nuclei_nonzero[axis] = pd.to_numeric(position_xyz[axis], errors='coerce')

print('temporary pt_position coordinate null counts:')
print(nuclei_nonzero[['x', 'y', 'z']].isna().sum().to_dict())


temporary pt_position coordinate null counts:
{'x': 0, 'y': 0, 'z': 0}


## Root-Level Aggregation

Aggregate the remaining nuclei rows by `pt_root_id`. The resulting root-level features include nucleus counts, unique supervoxel counts, volume summaries, and coordinate ranges used for exploratory vascular screening.

In [5]:
# Collapse nuclei rows into root-level features used for exploratory vascular screening.
root_nuclei_df = (
    nuclei_nonzero
    .groupby('pt_root_id')
    .agg(
        nuclei_rows=('id', 'size'),
        unique_supervoxels=('pt_supervoxel_id', 'nunique'),
        nucleus_volume_sum=('volume', 'sum'),
        nucleus_volume_max=('volume', 'max'),
        nucleus_volume_median=('volume', 'median'),
        x_min=('x', 'min'),
        x_max=('x', 'max'),
        y_min=('y', 'min'),
        y_max=('y', 'max'),
        z_min=('z', 'min'),
        z_max=('z', 'max'),
    )
    .reset_index()
)

root_nuclei_df['x_range'] = root_nuclei_df['x_max'] - root_nuclei_df['x_min']
root_nuclei_df['y_range'] = root_nuclei_df['y_max'] - root_nuclei_df['y_min']
root_nuclei_df['z_range'] = root_nuclei_df['z_max'] - root_nuclei_df['z_min']

ordered_columns = [
    'pt_root_id',
    'nuclei_rows',
    'unique_supervoxels',
    'nucleus_volume_sum',
    'nucleus_volume_max',
    'nucleus_volume_median',
    'x_min', 'x_max', 'x_range',
    'y_min', 'y_max', 'y_range',
    'z_min', 'z_max', 'z_range',
]
root_nuclei_df = root_nuclei_df[ordered_columns]

print(f"root_nuclei_df shape before annotation join: {root_nuclei_df.shape}")
display(root_nuclei_df.head())


root_nuclei_df shape before annotation join: (13873, 15)


,pt_root_id,nuclei_rows,unique_supervoxels,nucleus_volume_sum,nucleus_volume_max,nucleus_volume_median,x_min,x_max,x_range,y_min,y_max,y_range,z_min,z_max,z_range
0,648518346341364951,2,2,2.049131,1.511654,1.024566,64112,64112,0,32240,32240,0,1020,1037,17
1,648518346341389635,1,1,0.612127,0.612127,0.612127,65824,65824,0,41008,41008,0,101,101,0
2,648518346341403465,1,1,3.922837,3.922837,3.922837,65792,65792,0,40560,40560,0,113,113,0
3,648518346341404984,1,1,1.194394,1.194394,1.194394,70000,70000,0,60512,60512,0,1947,1947,0
4,648518346341407020,1,1,1.593769,1.593769,1.593769,65760,65760,0,41200,41200,0,188,188,0


## Add Annotation Reference Fields

Join selected fields from the local CA3 annotation CSV as reference metadata. These annotations help provide a neuronal comparison group later, but the CSV is not used as a vascular/nonvascular label source.

In [6]:
annotation_reference_columns = ['Cell ID']
for column in ['type', 'subtypes']:
    if column in annotation_df.columns:
        annotation_reference_columns.append(column)

annotation_reference_df = annotation_df[annotation_reference_columns].copy()
annotation_reference_df = annotation_reference_df.rename(columns={'Cell ID': 'pt_root_id'})
annotation_reference_df['pt_root_id'] = pd.to_numeric(annotation_reference_df['pt_root_id'], errors='coerce').astype('Int64')
annotation_reference_df = annotation_reference_df.dropna(subset=['pt_root_id']).drop_duplicates(subset=['pt_root_id'])

root_nuclei_with_annotations_df = root_nuclei_df.merge(
    annotation_reference_df,
    on='pt_root_id',
    how='left',
)
root_nuclei_with_annotations_df['annotation_match'] = root_nuclei_with_annotations_df['type'].notna() if 'type' in root_nuclei_with_annotations_df.columns else root_nuclei_with_annotations_df['pt_root_id'].isin(annotation_reference_df['pt_root_id'])

annotation_ordered_columns = ['pt_root_id', 'annotation_match']
annotation_ordered_columns += [column for column in ['type', 'subtypes'] if column in root_nuclei_with_annotations_df.columns]
annotation_ordered_columns += [
    column for column in root_nuclei_with_annotations_df.columns
    if column not in annotation_ordered_columns
]
root_nuclei_with_annotations_df = root_nuclei_with_annotations_df[annotation_ordered_columns]

print(f"root_nuclei_with_annotations_df shape: {root_nuclei_with_annotations_df.shape}")
print(f"annotation matches: {int(root_nuclei_with_annotations_df['annotation_match'].sum())}")
print(f"unmatched roots: {int((~root_nuclei_with_annotations_df['annotation_match']).sum())}")
display(root_nuclei_with_annotations_df.head())


root_nuclei_with_annotations_df shape: (13873, 18)
annotation matches: 2061
unmatched roots: 11812


,pt_root_id,annotation_match,type,subtypes,nuclei_rows,unique_supervoxels,nucleus_volume_sum,nucleus_volume_max,nucleus_volume_median,x_min,x_max,x_range,y_min,y_max,y_range,z_min,z_max,z_range
0,648518346341364951,False,NaN,NaN,2,2,2.049131,1.511654,1.024566,64112,64112,0,32240,32240,0,1020,1037,17
1,648518346341389635,False,NaN,NaN,1,1,0.612127,0.612127,0.612127,65824,65824,0,41008,41008,0,101,101,0
2,648518346341403465,False,NaN,NaN,1,1,3.922837,3.922837,3.922837,65792,65792,0,40560,40560,0,113,113,0
3,648518346341404984,False,NaN,NaN,1,1,1.194394,1.194394,1.194394,70000,70000,0,60512,60512,0,1947,1947,0
4,648518346341407020,False,NaN,NaN,1,1,1.593769,1.593769,1.593769,65760,65760,0,41200,41200,0,188,188,0


## Lightweight Summaries

These summaries show the root-level population before vascular screening, including annotation coverage, nucleus-count distribution, volume distribution, spatial spread, and the largest multi-nucleus roots.

In [7]:
print(f"root-level row count: {len(root_nuclei_with_annotations_df)}")
print(f"annotated root count: {int(root_nuclei_with_annotations_df['annotation_match'].sum())}")
print(f"unannotated root count: {int((~root_nuclei_with_annotations_df['annotation_match']).sum())}")

print('nuclei_rows descriptive stats:')
display(root_nuclei_with_annotations_df['nuclei_rows'].describe())

print('nucleus_volume_sum descriptive stats:')
display(root_nuclei_with_annotations_df['nucleus_volume_sum'].describe())

print('coordinate range descriptive stats:')
display(root_nuclei_with_annotations_df[['x_range', 'y_range', 'z_range']].describe())

print('top 20 roots by nuclei_rows:')
display(
    root_nuclei_with_annotations_df
    .sort_values(['nuclei_rows', 'nucleus_volume_sum'], ascending=False)
    .head(20)
)


root-level row count: 13873
annotated root count: 2061
unannotated root count: 11812
nuclei_rows descriptive stats:


count     13873.0
mean     1.748288
std      2.878962
min           1.0
25%           1.0
50%           1.0
75%           2.0
max         188.0
Name: nuclei_rows, dtype: Float64

nucleus_volume_sum descriptive stats:


count        13873.0
mean      201.201157
std       381.450775
min         0.011197
25%         0.705439
50%          8.03603
75%       169.995804
max      1965.434448
Name: nucleus_volume_sum, dtype: Float64

coordinate range descriptive stats:


,x_range,y_range,z_range
count,13873.000000,13873.000000,13873.000000
mean,183.937144,168.739854,98.034888
std,756.025839,668.824631,283.834901
min,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000
max,29072.000000,18864.000000,2037.000000


top 20 roots by nuclei_rows:


,pt_root_id,annotation_match,type,subtypes,nuclei_rows,unique_supervoxels,nucleus_volume_sum,nucleus_volume_max,nucleus_volume_median,x_min,x_max,x_range,y_min,y_max,y_range,z_min,z_max,z_range
11223,648518346447755155,False,NaN,NaN,188,184,1965.434448,126.971504,0.15863,31168,60240,29072,40256,59120,18864,104,2141,2037
8813,648518346442399751,False,NaN,NaN,153,151,1086.704102,90.452919,0.160497,40128,58480,18352,52192,65856,13664,112,2141,2029
13315,648518346458497167,False,NaN,NaN,62,62,770.708618,243.869049,0.477757,66848,68032,1184,46864,53536,6672,111,2126,2015
12740,648518346454066613,False,NaN,NaN,54,54,1065.731323,302.375671,2.685519,61600,67456,5856,30832,38560,7728,151,2020,1869
6223,648518346437928629,False,NaN,NaN,44,42,378.89151,79.490623,0.21835,28592,39040,10448,62048,74816,12768,196,1953,1757
11227,648518346447766769,False,NaN,NaN,37,37,433.165497,82.82373,0.276204,23872,29264,5392,62112,70752,8640,122,1584,1462
4857,648518346435381847,False,NaN,NaN,33,33,262.774048,69.797379,0.186624,29696,33520,3824,56960,62256,5296,112,1863,1751
9911,648518346444717749,False,NaN,NaN,32,31,103.621117,65.393051,0.115707,51952,62000,10048,68768,71152,2384,130,1892,1762
11795,648518346449540678,False,NaN,NaN,30,30,236.702682,62.474251,1.92596,52192,53248,1056,32880,33824,944,1216,2137,921
11393,648518346448223041,False,NaN,NaN,29,28,389.745544,78.800117,0.186624,59744,65040,5296,62112,70608,8496,171,2141,1970


## Likely Large Multi-Cell Roots

This exploratory table lists roots with unusually many associated nuclei. It is a candidate-generation view for closer inspection, not an automatic vascular call.

In [8]:
# Exploratory threshold used to identify high-duplication roots for closer inspection.
large_multicell_candidate_roots = (
    root_nuclei_with_annotations_df[root_nuclei_with_annotations_df['nuclei_rows'] >= 10]
    .sort_values(['nuclei_rows', 'nucleus_volume_sum'], ascending=False)
)

large_multicell_columns = [
    'pt_root_id',
    'nuclei_rows',
    'nucleus_volume_sum',
    'x_range',
    'y_range',
    'z_range',
    'annotation_match',
]
large_multicell_columns += [
    column for column in ['type', 'subtypes']
    if column in large_multicell_candidate_roots.columns
]

print(f"roots with nuclei_rows >= 10: {len(large_multicell_candidate_roots)}")
display(large_multicell_candidate_roots[large_multicell_columns])


roots with nuclei_rows >= 10: 135


,pt_root_id,nuclei_rows,nucleus_volume_sum,x_range,y_range,z_range,annotation_match,type,subtypes
11223,648518346447755155,188,1965.434448,29072,18864,2037,False,NaN,NaN
8813,648518346442399751,153,1086.704102,18352,13664,2029,False,NaN,NaN
13315,648518346458497167,62,770.708618,1184,6672,2015,False,NaN,NaN
12740,648518346454066613,54,1065.731323,5856,7728,1869,False,NaN,NaN
6223,648518346437928629,44,378.89151,10448,12768,1757,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...
7915,648518346440844507,10,49.671844,2192,3504,1976,False,NaN,NaN
10023,648518346444913333,10,15.825715,688,720,1747,False,NaN,NaN
3364,648518346431717636,10,14.44843,176,128,300,False,NaN,NaN
10764,648518346446743283,10,11.343007,1040,576,865,False,NaN,NaN


## Exploratory Curation Approach

Next, I compare manually reviewed examples against simple nuclei-derived features to build a small, conservative candidate set for visual inspection. The goal is to remove obvious vascular segmentations from the nonzero nuclei-associated root population before later neuron/glia exploration, not to classify every root biologically.

## Manual-reference comparison for vascular filtering

I manually inspected roots with unusually high nucleus counts and broad spatial extent and used those reviewed roots as vascular reference examples. I also used annotated neuronal roots as a nonvascular comparison group. These reference sets are for exploratory threshold testing only; they are not labels from an automated classifier.


In [9]:
# Manual vascular references from viewer inspection and annotated neuronal comparison roots.
manual_reference_high_duplication_root_ids = [
    648518346430972250,
    648518346431527510,
    648518346435381847,
    648518346437928629,
    648518346441638397,
    648518346442399751,
    648518346444231783,
    648518346444717749,
    648518346447755155,
    648518346447766769,
    648518346448223041,
    648518346449284164,
    648518346449540678,
    648518346454066613,
    648518346458497167,
]

manual_reference_annotated_neuronal_root_ids = [
    648518346448199666,
    648518346462161795,
    648518346443172484,
    648518346454880649,
    648518346438316205,
    648518346429301812,
    648518346443147652,
    648518346446645962,
    648518346446991251,
    648518346440948518,
    648518346448327390,
    648518346442391506,
    648518346442089733,
    648518346438029017,
    648518346448049658,
]

manual_reference_sets_df = pd.concat(
    [
        pd.DataFrame({
            'reference_set': 'high_duplication_manual_reference',
            'pt_root_id': pd.Series(manual_reference_high_duplication_root_ids, dtype='Int64'),
        }),
        pd.DataFrame({
            'reference_set': 'annotated_neuronal_manual_reference',
            'pt_root_id': pd.Series(manual_reference_annotated_neuronal_root_ids, dtype='Int64'),
        }),
    ],
    ignore_index=True,
)

root_level_aggregate_df = root_nuclei_with_annotations_df.copy()
root_level_aggregate_df['pt_root_id'] = pd.to_numeric(root_level_aggregate_df['pt_root_id'], errors='coerce').astype('Int64')

reference_feature_df = manual_reference_sets_df.merge(
    root_level_aggregate_df,
    on='pt_root_id',
    how='left',
)

reference_display_columns = [
    'reference_set',
    'pt_root_id',
    'nuclei_rows',
    'nucleus_volume_sum',
    'nucleus_volume_max',
    'x_range',
    'y_range',
    'z_range',
]
if 'annotation_match' in reference_feature_df.columns:
    reference_display_columns.append('annotation_match')

print('Manual reference feature values:')
display(reference_feature_df[reference_display_columns])

missing_reference_roots = reference_feature_df[reference_feature_df['nuclei_rows'].isna()][['reference_set', 'pt_root_id']]
if not missing_reference_roots.empty:
    print('Reference roots not present in the root-level aggregate table:')
    display(missing_reference_roots)



Manual reference feature values:


,reference_set,pt_root_id,nuclei_rows,nucleus_volume_sum,nucleus_volume_max,x_range,y_range,z_range,annotation_match
0,high_duplication_manual_reference,648518346430972250,25,393.817688,151.404312,5456,5072,2006,False
1,high_duplication_manual_reference,648518346431527510,27,659.182068,257.048431,6896,4176,1837,False
2,high_duplication_manual_reference,648518346435381847,33,262.774048,69.797379,3824,5296,1751,False
3,high_duplication_manual_reference,648518346437928629,44,378.89151,79.490623,10448,12768,1757,False
4,high_duplication_manual_reference,648518346441638397,28,542.400269,134.410339,8448,5856,2011,False
5,high_duplication_manual_reference,648518346442399751,153,1086.704102,90.452919,18352,13664,2029,False
6,high_duplication_manual_reference,648518346444231783,25,143.842316,74.440582,4096,4864,1594,False
7,high_duplication_manual_reference,648518346444717749,32,103.621117,65.393051,10048,2384,1762,False
8,high_duplication_manual_reference,648518346447755155,188,1965.434448,126.971504,29072,18864,2037,False
9,high_duplication_manual_reference,648518346447766769,37,433.165497,82.82373,5392,8640,1462,False


In [10]:
reference_summary_features = ['nuclei_rows', 'x_range', 'y_range', 'z_range']

reference_compact_summary_df = (
    reference_feature_df
    .groupby('reference_set')[reference_summary_features]
    .agg([
        ('n', 'count'),
        ('median', 'median'),
        ('p75', lambda s: s.quantile(0.75)),
        ('p90', lambda s: s.quantile(0.90)),
        ('p95', lambda s: s.quantile(0.95)),
        ('p99', lambda s: s.quantile(0.99)),
        ('max', 'max'),
    ])
)

print('Compact descriptive summaries for manual reference sets:')
display(reference_compact_summary_df)



Compact descriptive summaries for manual reference sets:


nuclei_rows                             \
                                              n median   p75    p90    p95   
reference_set                                                                
annotated_neuronal_manual_reference          15   14.0  14.5   16.0   16.0   
high_duplication_manual_reference            15   32.0  49.0  116.6  163.5   

                                                x_range                  ...  \
                                       p99  max       n  median     p75  ...   
reference_set                                                            ...   
annotated_neuronal_manual_reference   16.0   16      15   720.0  2192.0  ...   
high_duplication_manual_reference    183.1  188      15  5456.0  9248.0  ...   

                                     y_range                  z_range          \
                                         p95       p99    max       n  median   
reference_set                                                                   
annotated_neuronal_manual_reference   5606.4  10068.48  11184      15   677.0   
high_duplication_manual_reference    15224.0  18136.00  18864      15  1837.0   

                                                                            
                                        p75     p90     p95      p99   max  
reference_set                                                               
annotated_neuronal_manual_reference   768.0  1022.0  1168.6  1438.52  1506  
high_duplication_manual_reference    2008.5  2023.4  2031.4  2035.88  2037  

[2 rows x 28 columns]

In [11]:
percentile_rank_features = ['nuclei_rows', 'x_range', 'y_range', 'z_range']
percentile_rank_columns = []

percentile_rank_df = root_level_aggregate_df[['pt_root_id'] + percentile_rank_features].copy()
for feature in percentile_rank_features:
    percentile_column = f'{feature}_percentile_rank'
    percentile_rank_columns.append(percentile_column)
    percentile_rank_df[percentile_column] = percentile_rank_df[feature].rank(method='max', pct=True) * 100

reference_percentile_rank_df = manual_reference_sets_df.merge(
    percentile_rank_df[['pt_root_id'] + percentile_rank_columns],
    on='pt_root_id',
    how='left',
)

print('Manual reference percentile ranks against the full nonzero-root distribution:')
display(reference_percentile_rank_df)



Manual reference percentile ranks against the full nonzero-root distribution:


,reference_set,pt_root_id,nuclei_rows_percentile_rank,x_range_percentile_rank,y_range_percentile_rank,z_range_percentile_rank
0,high_duplication_manual_reference,648518346430972250,99.906293,99.697254,99.668421,99.942334
1,high_duplication_manual_reference,648518346431527510,99.913501,99.827002,99.545880,99.596338
2,high_duplication_manual_reference,648518346435381847,99.956751,99.286384,99.682837,99.394507
3,high_duplication_manual_reference,648518346437928629,99.971167,99.963959,99.963959,99.408924
4,high_duplication_manual_reference,648518346441638397,99.927918,99.906293,99.769336,99.956751
5,high_duplication_manual_reference,648518346442399751,99.992792,99.978375,99.978375,99.985584
6,high_duplication_manual_reference,648518346444231783,99.906293,99.365674,99.646796,98.998054
7,high_duplication_manual_reference,648518346444717749,99.949542,99.942334,98.536726,99.430549
8,high_duplication_manual_reference,648518346447755155,100.000000,100.000000,100.000000,100.000000
9,high_duplication_manual_reference,648518346447766769,99.963959,99.682837,99.920709,98.616017


In [12]:
high_dup_reference = reference_feature_df[
    reference_feature_df['reference_set'] == 'high_duplication_manual_reference'
].copy()
neuron_reference = reference_feature_df[
    reference_feature_df['reference_set'] == 'annotated_neuronal_manual_reference'
].copy()

high_dup_medians = high_dup_reference[reference_summary_features].median(numeric_only=True)
neuron_medians = neuron_reference[reference_summary_features].median(numeric_only=True)

high_dup_p75_exceeds_neuron = bool(
    (high_dup_medians[['nuclei_rows', 'x_range', 'y_range', 'z_range']] >
     neuron_medians[['nuclei_rows', 'x_range', 'y_range', 'z_range']]).all()
)

print('Factual exploratory summary:')
print(
    f"High-duplication manual references present in aggregate table: "
    f"{int(high_dup_reference['nuclei_rows'].notna().sum())} of {len(manual_reference_high_duplication_root_ids)}."
)
print(
    f"Annotated neuronal manual references present in aggregate table: "
    f"{int(neuron_reference['nuclei_rows'].notna().sum())} of {len(manual_reference_annotated_neuronal_root_ids)}."
)
print('Median feature values by reference set:')
display(pd.DataFrame({
    'high_duplication_manual_reference_median': high_dup_medians,
    'annotated_neuronal_manual_reference_median': neuron_medians,
}))

if high_dup_p75_exceeds_neuron:
    print(
        'These manually inspected groups show separation in nucleus count and spatial-range medians, '
        'which is enough to justify testing a simple conservative vascular/problematic-root rule. '
        'No rule is applied here.'
    )
else:
    print(
        'These manually inspected groups should be reviewed with the percentile-rank table before choosing thresholds; '
        'the section is sufficient for exploratory rule testing, but no rule is applied here.'
    )



Factual exploratory summary:
High-duplication manual references present in aggregate table: 15 of 15.
Annotated neuronal manual references present in aggregate table: 15 of 15.
Median feature values by reference set:


,high_duplication_manual_reference_median,annotated_neuronal_manual_reference_median
nuclei_rows,32.0,14.0
x_range,5456.0,720.0
y_range,5856.0,672.0
z_range,1837.0,677.0


These manually inspected groups show separation in nucleus count and spatial-range medians, which is enough to justify testing a simple conservative vascular/problematic-root rule. No rule is applied here.


## Evaluate conservative vascular/problematic-root rules

Evaluate simple percentile-based candidate rules against the full nonzero-root distribution and the two manual-reference groups. The rules combine high nucleus count with unusually large spatial spread to produce manageable candidate sets for manual inspection; no roots are excluded in this section.


In [13]:
# Conservative exploratory thresholds for generating a small manual-inspection candidate set.
spread_features = ['x_range', 'y_range', 'z_range']
rule_base_feature = 'nuclei_rows'
rule_base_threshold = 25
spread_percentile_levels = [0.99, 0.995]

spread_thresholds = {
    percentile_level: root_level_aggregate_df[spread_features].quantile(percentile_level)
    for percentile_level in spread_percentile_levels
}

print('Spatial-range thresholds from full nonzero-root distribution:')
display(
    pd.DataFrame(spread_thresholds)
    .rename(columns={0.99: 'p99', 0.995: 'p99_5'})
)

rule_evaluation_df = root_level_aggregate_df.copy()
for percentile_level, thresholds in spread_thresholds.items():
    suffix = str(percentile_level).replace('.', '_')
    for feature in spread_features:
        rule_evaluation_df[f'{feature}_gte_p{suffix}'] = rule_evaluation_df[feature] >= thresholds[feature]
    rule_evaluation_df[f'spread_features_gte_p{suffix}_count'] = rule_evaluation_df[
        [f'{feature}_gte_p{suffix}' for feature in spread_features]
    ].sum(axis=1)

rule_definitions = [
    {
        'rule_id': 'A',
        'rule_description': 'nuclei_rows >= 25 AND at least one spatial-range feature >= 99th percentile',
        'mask': (
            (rule_evaluation_df[rule_base_feature] >= rule_base_threshold) &
            (rule_evaluation_df['spread_features_gte_p0_99_count'] >= 1)
        ),
    },
    {
        'rule_id': 'B',
        'rule_description': 'nuclei_rows >= 25 AND at least two spatial-range features >= 99th percentile',
        'mask': (
            (rule_evaluation_df[rule_base_feature] >= rule_base_threshold) &
            (rule_evaluation_df['spread_features_gte_p0_99_count'] >= 2)
        ),
    },
    {
        'rule_id': 'C',
        'rule_description': 'nuclei_rows >= 25 AND z_range >= 99th percentile',
        'mask': (
            (rule_evaluation_df[rule_base_feature] >= rule_base_threshold) &
            rule_evaluation_df['z_range_gte_p0_99']
        ),
    },
    {
        'rule_id': 'D',
        'rule_description': 'nuclei_rows >= 25 AND at least one spatial-range feature >= 99.5th percentile',
        'mask': (
            (rule_evaluation_df[rule_base_feature] >= rule_base_threshold) &
            (rule_evaluation_df['spread_features_gte_p0_995_count'] >= 1)
        ),
    },
]

high_duplication_reference_ids = set(manual_reference_high_duplication_root_ids)
annotated_neuronal_reference_ids = set(manual_reference_annotated_neuronal_root_ids)
total_roots = len(rule_evaluation_df)

rule_result_rows = []
for rule_definition in rule_definitions:
    flagged_roots = rule_evaluation_df.loc[rule_definition['mask'], 'pt_root_id'].astype('Int64')
    flagged_root_ids = set(flagged_roots.dropna().astype(int).tolist())
    missed_high_duplication_ids = sorted(high_duplication_reference_ids - flagged_root_ids)
    flagged_annotated_neuronal_ids = sorted(annotated_neuronal_reference_ids & flagged_root_ids)

    rule_result_rows.append({
        'rule_id': rule_definition['rule_id'],
        'rule_description': rule_definition['rule_description'],
        'candidate_label': 'vascular_problematic_candidate',
        'total_roots_flagged': len(flagged_root_ids),
        'percent_all_roots_flagged': 100 * len(flagged_root_ids) / total_roots,
        'high_duplication_references_flagged': len(high_duplication_reference_ids & flagged_root_ids),
        'annotated_neuronal_references_flagged': len(flagged_annotated_neuronal_ids),
        'missed_high_duplication_reference_ids': missed_high_duplication_ids,
        'flagged_annotated_neuronal_reference_ids': flagged_annotated_neuronal_ids,
    })

candidate_rule_results_df = pd.DataFrame(rule_result_rows)

print('Candidate conservative vascular/problematic-root rule evaluation:')
display(candidate_rule_results_df)

compact_rule_comparison_df = candidate_rule_results_df[[
    'rule_id',
    'total_roots_flagged',
    'percent_all_roots_flagged',
    'high_duplication_references_flagged',
    'annotated_neuronal_references_flagged',
]].copy()
compact_rule_comparison_df['percent_all_roots_flagged'] = compact_rule_comparison_df['percent_all_roots_flagged'].round(3)

print('Compact candidate-rule comparison:')
display(compact_rule_comparison_df)



Spatial-range thresholds from full nonzero-root distribution:


,p99,p99_5
x_range,3236.48,4730.24
y_range,2740.48,3786.24
z_range,1594.56,1796.48


Candidate conservative vascular/problematic-root rule evaluation:


,rule_id,rule_description,candidate_label,total_roots_flagged,percent_all_roots_flagged,high_duplication_references_flagged,annotated_neuronal_references_flagged,missed_high_duplication_reference_ids,flagged_annotated_neuronal_reference_ids
0,A,nuclei_rows >= 25 AND at least one spatial-ran...,vascular_problematic_candidate,14,0.100915,14,0,[648518346449540678],[]
1,B,nuclei_rows >= 25 AND at least two spatial-ran...,vascular_problematic_candidate,13,0.093707,13,0,"[648518346449284164, 648518346449540678]",[]
2,C,nuclei_rows >= 25 AND z_range >= 99th percentile,vascular_problematic_candidate,11,0.079291,11,0,"[648518346444231783, 648518346447766769, 64851...",[]
3,D,nuclei_rows >= 25 AND at least one spatial-ran...,vascular_problematic_candidate,13,0.093707,13,0,"[648518346449284164, 648518346449540678]",[]


Compact candidate-rule comparison:


,rule_id,total_roots_flagged,percent_all_roots_flagged,high_duplication_references_flagged,annotated_neuronal_references_flagged
0,A,14,0.101,14,0
1,B,13,0.094,13,0
2,C,11,0.079,11,0
3,D,13,0.094,13,0


In [14]:
print(candidate_rule_results_df.missed_high_duplication_reference_ids.to_list())

[[648518346449540678], [648518346449284164, 648518346449540678], [648518346444231783, 648518346447766769, 648518346449284164, 648518346449540678], [648518346449284164, 648518346449540678]]


In [15]:
recommendation_candidates_df = candidate_rule_results_df.copy()
recommendation_candidates_df['missed_high_duplication_reference_count'] = (
    len(high_duplication_reference_ids) - recommendation_candidates_df['high_duplication_references_flagged']
)
recommendation_candidates_df['conservatism_sort_key'] = list(zip(
    recommendation_candidates_df['annotated_neuronal_references_flagged'],
    recommendation_candidates_df['total_roots_flagged'],
    recommendation_candidates_df['missed_high_duplication_reference_count'],
))

best_rule_row = (
    recommendation_candidates_df
    .sort_values(
        [
            'annotated_neuronal_references_flagged',
            'total_roots_flagged',
            'missed_high_duplication_reference_count',
        ],
        ascending=[True, True, True],
    )
    .iloc[0]
)

print('Concise recommendation:')
print(
    f"Rule {best_rule_row['rule_id']} appears most conservative among the tested candidates: "
    f"it flags {int(best_rule_row['total_roots_flagged'])} total roots "
    f"({best_rule_row['percent_all_roots_flagged']:.3f}% of all roots), captures "
    f"{int(best_rule_row['high_duplication_references_flagged'])} of {len(high_duplication_reference_ids)} "
    f"high-duplication manual-reference roots, and flags "
    f"{int(best_rule_row['annotated_neuronal_references_flagged'])} of {len(annotated_neuronal_reference_ids)} "
    f"annotated-neuronal reference roots. Treat this only as a candidate rule for testing; no roots are excluded here."
)



Concise recommendation:
Rule C appears most conservative among the tested candidates: it flags 11 total roots (0.079% of all roots), captures 11 of 15 high-duplication manual-reference roots, and flags 0 of 15 annotated-neuronal reference roots. Treat this only as a candidate rule for testing; no roots are excluded here.


## Compare Rule B and Rule D flagged roots

Compare only the existing candidate Rule B and Rule D flagged root sets before choosing which roots need manual inspection. The printed root-ID lists are intended to support manual review in Neuroglancer/Spelunker or a similar viewer.

The previous diagnostic identifies Rule C as numerically most conservative because it flags fewer roots. This workflow proceeds with Rule B because Rule B and Rule D converge on the same practical 13-root candidate set used for final manual review.


In [16]:
# Candidate root IDs used for Rule B / Rule D comparison and manual viewer inspection.
rule_masks_by_id = {
    rule_definition['rule_id']: rule_definition['mask']
    for rule_definition in rule_definitions
}

rule_b_flagged_ids = sorted(
    rule_evaluation_df.loc[rule_masks_by_id['B'], 'pt_root_id']
    .dropna()
    .astype(int)
    .tolist()
)
rule_d_flagged_ids = sorted(
    rule_evaluation_df.loc[rule_masks_by_id['D'], 'pt_root_id']
    .dropna()
    .astype(int)
    .tolist()
)

rule_b_flagged_set = set(rule_b_flagged_ids)
rule_d_flagged_set = set(rule_d_flagged_ids)

rule_b_and_d_ids = sorted(rule_b_flagged_set & rule_d_flagged_set)
rule_b_only_ids = sorted(rule_b_flagged_set - rule_d_flagged_set)
rule_d_only_ids = sorted(rule_d_flagged_set - rule_b_flagged_set)
symmetric_difference_ids = sorted(rule_b_flagged_set ^ rule_d_flagged_set)

print(f"number flagged by B: {len(rule_b_flagged_ids)}")
print(f"number flagged by D: {len(rule_d_flagged_ids)}")
print(f"number flagged by both: {len(rule_b_and_d_ids)}")
print(f"root IDs flagged by both: {rule_b_and_d_ids}")
print(f"root IDs unique to B: {rule_b_only_ids}")
print(f"root IDs unique to D: {rule_d_only_ids}")

print('Compact Python list for unique-to-B IDs:')
print(rule_b_only_ids)
print('Compact Python list for unique-to-D IDs:')
print(rule_d_only_ids)

symmetric_difference_columns = [
    'pt_root_id',
    'nuclei_rows',
    'x_range',
    'y_range',
    'z_range',
    'nucleus_volume_sum',
]

symmetric_difference_comparison_df = (
    rule_evaluation_df[rule_evaluation_df['pt_root_id'].isin(symmetric_difference_ids)]
    [symmetric_difference_columns]
    .copy()
    .sort_values('pt_root_id')
)

print('Rule B vs Rule D symmetric-difference root features:')
display(symmetric_difference_comparison_df)

print('Brief recommendation:')
if rule_b_flagged_set == rule_d_flagged_set:
    print('Rule B and Rule D flag identical root sets.')
else:
    print(
        'Rule B and Rule D differ; manually inspect only the differing roots listed above before choosing between them.'
    )



number flagged by B: 13
number flagged by D: 13
number flagged by both: 13
root IDs flagged by both: [648518346430972250, 648518346431527510, 648518346435381847, 648518346437928629, 648518346441638397, 648518346442399751, 648518346444231783, 648518346444717749, 648518346447755155, 648518346447766769, 648518346448223041, 648518346454066613, 648518346458497167]
root IDs unique to B: []
root IDs unique to D: []
Compact Python list for unique-to-B IDs:
[]
Compact Python list for unique-to-D IDs:
[]
Rule B vs Rule D symmetric-difference root features:


,pt_root_id,nuclei_rows,x_range,y_range,z_range,nucleus_volume_sum


Brief recommendation:
Rule B and Rule D flag identical root sets.


## Final conservative vascular screen

This final section formalizes the selected conservative high-precision vascular screen as `vascular_candidate`. The flag is intended to capture obvious vascular-like root objects using high nucleus duplication plus broad spatial spread. It is not a claim that all remaining roots are nonvascular; smaller vascular segments or ambiguous objects may remain after screening.

Selected rule: `nuclei_rows >= 25` and at least two of `x_range`, `y_range`, and `z_range` are greater than or equal to their 99th-percentile thresholds calculated from the full nonzero root-level dataset. This is the previously evaluated Rule B and is used to reproduce the candidate set I manually reviewed.


In [17]:
final_spread_features = ['x_range', 'y_range', 'z_range']
final_nuclei_rows_threshold = 25
final_spread_percentile = 0.99

final_root_feature_df = root_nuclei_with_annotations_df.copy()
final_root_feature_df['pt_root_id'] = pd.to_numeric(
    final_root_feature_df['pt_root_id'],
    errors='coerce',
).astype('Int64')
final_root_feature_df = final_root_feature_df[final_root_feature_df['pt_root_id'] != 0].copy()

final_spread_thresholds = (
    final_root_feature_df[final_spread_features]
    .quantile(final_spread_percentile)
    .to_dict()
)

final_spread_threshold_series = pd.Series(final_spread_thresholds)
final_root_feature_df['spatial_ranges_gte_99th_count'] = (
    final_root_feature_df[final_spread_features]
    .ge(final_spread_threshold_series)
    .sum(axis=1)
)

final_root_feature_df['vascular_candidate'] = (
    (final_root_feature_df['nuclei_rows'] >= final_nuclei_rows_threshold) &
    (final_root_feature_df['spatial_ranges_gte_99th_count'] >= 2)
)

final_rule_description = (
    'vascular_candidate == True when nuclei_rows >= 25 and at least 2 of '
    'x_range, y_range, and z_range are >= their 99th-percentile thresholds '
    'calculated from the full nonzero root-level dataset.'
)

final_feature_priority_columns = [
    'pt_root_id',
    'nuclei_rows',
    'unique_supervoxels',
    'nucleus_volume_sum',
    'nucleus_volume_max',
    'nucleus_volume_median',
    'x_range',
    'y_range',
    'z_range',
    'spatial_ranges_gte_99th_count',
    'vascular_candidate',
]
final_feature_priority_columns = [
    column for column in final_feature_priority_columns
    if column in final_root_feature_df.columns
]
final_root_feature_df = final_root_feature_df[
    final_feature_priority_columns + [
        column for column in final_root_feature_df.columns
        if column not in final_feature_priority_columns
    ]
]

print('Final vascular_candidate rule:')
print(final_rule_description)
print('99th-percentile spatial thresholds:')
display(pd.DataFrame([final_spread_thresholds], index=['threshold']))

print('Final root-level feature table preview:')
display(final_root_feature_df.head())



Final vascular_candidate rule:
vascular_candidate == True when nuclei_rows >= 25 and at least 2 of x_range, y_range, and z_range are >= their 99th-percentile thresholds calculated from the full nonzero root-level dataset.
99th-percentile spatial thresholds:


,x_range,y_range,z_range
threshold,3236.48,2740.48,1594.56


Final root-level feature table preview:


,pt_root_id,nuclei_rows,unique_supervoxels,nucleus_volume_sum,nucleus_volume_max,nucleus_volume_median,x_range,y_range,z_range,spatial_ranges_gte_99th_count,vascular_candidate,annotation_match,type,subtypes,x_min,x_max,y_min,y_max,z_min,z_max
0,648518346341364951,2,2,2.049131,1.511654,1.024566,0,0,17,0,False,False,NaN,NaN,64112,64112,32240,32240,1020,1037
1,648518346341389635,1,1,0.612127,0.612127,0.612127,0,0,0,0,False,False,NaN,NaN,65824,65824,41008,41008,101,101
2,648518346341403465,1,1,3.922837,3.922837,3.922837,0,0,0,0,False,False,NaN,NaN,65792,65792,40560,40560,113,113
3,648518346341404984,1,1,1.194394,1.194394,1.194394,0,0,0,0,False,False,NaN,NaN,70000,70000,60512,60512,1947,1947
4,648518346341407020,1,1,1.593769,1.593769,1.593769,0,0,0,0,False,False,NaN,NaN,65760,65760,41200,41200,188,188


## Manual vascular curation

I manually inspected the 13 final candidate roots in the Pyr.ai volume using viewer-based visual review. I retained roots that were clearly vascular and rejected candidates that instead looked like astrocytes, neuronal segmentations/fragments, or otherwise did not visually support a vascular interpretation.

The final vascular list therefore reflects manual visual curation informed by the exploratory screen. The screened nonvascular working dataset is created by removing those curated vascular roots from the nonzero nuclei-associated root population; "nonvascular" here means the obvious vascular roots identified by this workflow have been removed, not that every remaining root has received a complete biological cell-type classification.

In [18]:
# Vascular roots confirmed by manual visual inspection.
manually_reviewed_vascular_candidate_ids = sorted([
    648518346430972250,
    648518346431527510,
    648518346435381847,
    648518346437928629,
    648518346441638397,
    648518346442399751,
    648518346444231783,
    648518346444717749,
    648518346447755155,
    648518346447766769,
    648518346448223041,
    648518346454066613,
    648518346458497167,
])

vascular_candidate_ids = sorted(
    final_root_feature_df.loc[final_root_feature_df['vascular_candidate'], 'pt_root_id']
    .dropna()
    .astype(int)
    .tolist()
)

print(f'vascular_candidate roots identified: {len(vascular_candidate_ids)}')
print('vascular_candidate root IDs:')
print(vascular_candidate_ids)
print(
    'vascular_candidate IDs match the 13 manually reviewed vascular candidate roots: '
    f'{vascular_candidate_ids == manually_reviewed_vascular_candidate_ids}'
)

if vascular_candidate_ids != manually_reviewed_vascular_candidate_ids:
    print('IDs flagged but not in the manually reviewed vascular candidate list:')
    print(sorted(set(vascular_candidate_ids) - set(manually_reviewed_vascular_candidate_ids)))
    print('Manually reviewed vascular candidate IDs not flagged:')
    print(sorted(set(manually_reviewed_vascular_candidate_ids) - set(vascular_candidate_ids)))

print('Final vascular_candidate root feature rows:')
display(
    final_root_feature_df.loc[
        final_root_feature_df['vascular_candidate'],
        [
            'pt_root_id',
            'nuclei_rows',
            'unique_supervoxels',
            'nucleus_volume_sum',
            'nucleus_volume_max',
            'nucleus_volume_median',
            'x_range',
            'y_range',
            'z_range',
            'spatial_ranges_gte_99th_count',
            'vascular_candidate',
        ],
    ]
    .sort_values('pt_root_id')
)

vascular_candidate roots identified: 13
vascular_candidate root IDs:
[648518346430972250, 648518346431527510, 648518346435381847, 648518346437928629, 648518346441638397, 648518346442399751, 648518346444231783, 648518346444717749, 648518346447755155, 648518346447766769, 648518346448223041, 648518346454066613, 648518346458497167]
vascular_candidate IDs match the 13 manually reviewed vascular candidate roots: True
Final vascular_candidate root feature rows:


,pt_root_id,nuclei_rows,unique_supervoxels,nucleus_volume_sum,nucleus_volume_max,nucleus_volume_median,x_range,y_range,z_range,spatial_ranges_gte_99th_count,vascular_candidate
3073,648518346430972250,25,25,393.817688,151.404312,0.776356,5456,5072,2006,3,True
3264,648518346431527510,27,27,659.182068,257.048431,0.160497,6896,4176,1837,3,True
4857,648518346435381847,33,33,262.774048,69.797379,0.186624,3824,5296,1751,3,True
6223,648518346437928629,44,42,378.89151,79.490623,0.21835,10448,12768,1757,3,True
8330,648518346441638397,28,28,542.400269,134.410339,0.296732,8448,5856,2011,3,True
8813,648518346442399751,153,151,1086.704102,90.452919,0.160497,18352,13664,2029,3,True
9658,648518346444231783,25,25,143.842316,74.440582,0.186624,4096,4864,1594,2,True
9911,648518346444717749,32,31,103.621117,65.393051,0.115707,10048,2384,1762,2,True
11223,648518346447755155,188,184,1965.434448,126.971504,0.15863,29072,18864,2037,3,True
11227,648518346447766769,37,37,433.165497,82.82373,0.276204,5392,8640,1462,2,True


## Save root-level outputs

`screened_root_df` keeps roots where `vascular_candidate == False`. This screened subset is not a definitive nonvascular dataset; it is only the root-level table after removing the conservative vascular candidates identified above.

The cells below produce local/generated root-level feature artifacts, a screened root table for downstream analysis, and metadata. When `overwrite_existing` is `False`, existing artifacts are preserved instead of overwritten.

In [19]:
# Remove the curated vascular roots to create the screened nonvascular working population.
screened_root_df = final_root_feature_df[~final_root_feature_df['vascular_candidate']].copy()

total_nonzero_roots = len(final_root_feature_df)
vascular_candidate_count = int(final_root_feature_df['vascular_candidate'].sum())
screened_root_count = len(screened_root_df)

print(f'total nonzero roots: {total_nonzero_roots}')
print(f'vascular candidates: {vascular_candidate_count}')
print(f'retained screened roots: {screened_root_count}')



total nonzero roots: 13873
vascular candidates: 13
retained screened roots: 13860


In [20]:
import json

nuclei_output_dir = project_root / 'data' / 'nuclei'
root_features_output_path = nuclei_output_dir / 'c3_nuclei_root_features_mat195.parquet'
screened_root_features_output_path = nuclei_output_dir / 'c3_nuclei_root_features_screened_mat195.parquet'
metadata_output_path = nuclei_output_dir / 'c3_nuclei_root_features_mat195_metadata.json'

output_paths = [
    root_features_output_path,
    screened_root_features_output_path,
    metadata_output_path,
]
existing_output_paths = [path for path in output_paths if path.exists()]

root_feature_metadata = {
    'source_nuclei_table': str(nuclei_parquet_path),
    'materialization_version': 195,
    'vascular_rule_definition': final_rule_description,
    'vascular_rule_parameters': {
        'nuclei_rows_minimum': final_nuclei_rows_threshold,
        'spatial_range_percentile': final_spread_percentile,
        'minimum_spatial_features_at_or_above_threshold': 2,
        'spatial_thresholds': {
            feature: float(threshold)
            for feature, threshold in final_spread_thresholds.items()
        },
    },
    'total_roots': int(total_nonzero_roots),
    'vascular_candidate_count': int(vascular_candidate_count),
    'screened_root_count': int(screened_root_count),
    'output_filenames': {
        'root_features': root_features_output_path.name,
        'screened_root_features': screened_root_features_output_path.name,
        'metadata': metadata_output_path.name,
    },
}

if existing_output_paths and not overwrite_existing:
    print('Existing nuclei curation output found; keeping existing output set.')
    for existing_output_path in existing_output_paths:
        print_path('Kept output', existing_output_path, project_root, show_full_path)
    missing_output_paths = [path for path in output_paths if not path.exists()]
    if missing_output_paths:
        print('Output set is incomplete; no missing output files were created because overwrite_existing is False.')
        for missing_output_path in missing_output_paths:
            print_path('Missing output', missing_output_path, project_root, show_full_path)
else:
    final_root_feature_df.to_parquet(root_features_output_path, index=False)
    screened_root_df.to_parquet(screened_root_features_output_path, index=False)
    metadata_output_path.write_text(json.dumps(root_feature_metadata, indent=2), encoding='utf-8')

    print('Saved root-level feature table:')
    print(format_path(root_features_output_path, project_root, show_full_path))
    print('Saved screened root-level feature table:')
    print(format_path(screened_root_features_output_path, project_root, show_full_path))
    print('Saved metadata:')
    print(format_path(metadata_output_path, project_root, show_full_path))



Existing nuclei curation output found; keeping existing output set.
Kept output: data\nuclei\c3_nuclei_root_features_mat195.parquet
Kept output: data\nuclei\c3_nuclei_root_features_screened_mat195.parquet
Kept output: data\nuclei\c3_nuclei_root_features_mat195_metadata.json


## Closing note

This screen removes only obvious vascular-like roots. Smaller vascular segments may remain in the screened subset, and the absence of `vascular_candidate` should not be interpreted as a definitive nonvascular label. The screened subset is intended as the starting point for Notebook 06.
